# SPAM MAIL DETECTION

In [209]:
import numpy as np
import pandas as pd
import re
import nltk
import pickle as pk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

In [210]:
mail_data=pd.read_csv('spam.csv',encoding='latin-1')

In [211]:
mail_data.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [212]:
mail_data.isnull().sum()

v1               0
v2               0
Unnamed: 2    5522
Unnamed: 3    5560
Unnamed: 4    5566
dtype: int64

In [213]:
mail_data.shape

(5572, 5)

In [214]:
for column in mail_data.columns:
    print(f"Column: {column}")
    print(len(mail_data[column].unique()))


Column: v1
2
Column: v2
5169
Column: Unnamed: 2
44
Column: Unnamed: 3
11
Column: Unnamed: 4
6


In [215]:
new_mail_data = mail_data.where((pd.notnull(mail_data)), '')

In [216]:
new_mail_data.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",,,
1,ham,Ok lar... Joking wif u oni...,,,
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,,,
3,ham,U dun say so early hor... U c already then say...,,,
4,ham,"Nah I don't think he goes to usf, he lives aro...",,,


### Label Encoding

In [217]:
new_mail_data.loc[new_mail_data['v1'] == 'spam', 'v1'] = 0
new_mail_data.loc[new_mail_data['v1'] == 'ham', 'v1'] = 1

In [218]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [219]:
ps = PorterStemmer()

In [220]:
stop_words = set(stopwords.words('english'))

In [221]:
print(stopwords.words('english')[:10])

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an']


In [222]:
def preprocess_text(text):
    text = text.lower()  
    text = re.sub(r'http\S+', ' ', text)   
    text = re.sub(r'[^a-zA-Z]', ' ', text)   
    words = text.split()    
    words = [word for word in words if word not in stop_words]    
    words = [ps.stem(word) for word in words]    
    return " ".join(words)

In [223]:
new_mail_data['processed_text'] = new_mail_data['v2'].apply(preprocess_text)

spam - 0<br>
ham - 1

In [224]:
X = new_mail_data['processed_text']
Y = new_mail_data['v1']

In [225]:
print(X)

0       go jurong point crazi avail bugi n great world...
1                                   ok lar joke wif u oni
2       free entri wkli comp win fa cup final tkt st m...
3                     u dun say earli hor u c alreadi say
4                    nah think goe usf live around though
                              ...                        
5567    nd time tri contact u u pound prize claim easi...
5568                                b go esplanad fr home
5569                                    piti mood suggest
5570    guy bitch act like interest buy someth els nex...
5571                                       rofl true name
Name: processed_text, Length: 5572, dtype: object


In [226]:
print(Y)

0       1
1       1
2       0
3       1
4       1
       ..
5567    0
5568    1
5569    1
5570    1
5571    1
Name: v1, Length: 5572, dtype: object


In [227]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=3)

In [228]:
print(X.shape)
print(X_train.shape)
print(X_test.shape)

(5572,)
(4457,)
(1115,)


In [229]:
vectorizer = TfidfVectorizer(min_df=1, stop_words='english', lowercase=True)
X_train_features = vectorizer.fit_transform(X_train)
X_test_features = vectorizer.transform(X_test)


In [230]:
print(X_train)

3075    mum hope great day hope text meet well full li...
1787                                   ye sura sun tv lol
1614                     sef dey laugh meanwhil darl anji
4304                                   yo come carlo soon
3266                               ok come n pick u engin
                              ...                        
789                            gud mrng dear hav nice day
968                                 will go aptitud class
1667           dad gonna call get work ask crazi question
3321    ok darlin supos ok worri much film stuff mate ...
1688                           nan sonathaya soladha boss
Name: processed_text, Length: 4457, dtype: object


In [231]:
print(X_train_features)

  (0, 3056)	0.3179757859420271
  (0, 2144)	0.4646590529643843
  (0, 1941)	0.48097307958182434
  (0, 1116)	0.39791182613570947
  (0, 4735)	0.2122192373027003
  (0, 2870)	0.23473536171389747
  (0, 2635)	0.2634361483960197
  (0, 11)	0.35182117672148316
  (1, 5420)	0.35056971070320353
  (1, 4605)	0.5652509076654626
  (1, 4582)	0.4769136859540388
  (1, 4954)	0.4306015894277422
  (1, 2690)	0.380431198316959
  (2, 4106)	0.45329962489137066
  (2, 1195)	0.3811461667946165
  (2, 2585)	0.34506943774623944
  (2, 2862)	0.41722289584299355
  (2, 1106)	0.3880961710740478
  (2, 176)	0.45329962489137066
  (3, 5442)	0.524841815062256
  (3, 900)	0.34970933876863236
  (3, 709)	0.5979251862237673
  (3, 4371)	0.49470184881344015
  (4, 900)	0.3426740073688487
  (4, 3280)	0.3468819100246315
  :	:
  (4452, 3032)	0.4992311758162137
  (4452, 1123)	0.3469396289230092
  (4452, 2040)	0.4623655053601519
  (4452, 3159)	0.40546348662124637
  (4453, 840)	0.5555413757894101
  (4453, 230)	0.8314888933629899
  (4454, 3762

In [232]:
Y_train = Y_train.astype('int')
Y_test = Y_test.astype('int')

In [233]:
smote = SMOTE(random_state=42)

In [234]:
X_train_smote, Y_train_smote = smote.fit_resample(X_train_features, Y_train)

In [235]:
print(Y_train_smote.value_counts())

v1
1    3865
0    3865
Name: count, dtype: int64


In [236]:
models = {
    "Logistic Regression": LogisticRegression(),
    "SVM": LinearSVC(),
}

In [237]:
for name, model in models.items():

    print("="*60)
    print(name)
    print("="*60)

    model.fit(X_train_smote, Y_train_smote)

    train_pred = model.predict(X_train_features)

    print("\nTRAINING RESULTS")
    print("Accuracy:", accuracy_score(Y_train, train_pred))
    print("Confusion Matrix:\n", confusion_matrix(Y_train, train_pred))
    print("Classification Report:\n", classification_report(Y_train, train_pred))

    test_pred = model.predict(X_test_features)

    print("\nTEST RESULTS")
    print("Accuracy:", accuracy_score(Y_test, test_pred))
    print("Confusion Matrix:\n", confusion_matrix(Y_test, test_pred))
    print("Classification Report:\n", classification_report(Y_test, test_pred))

    print("\n\n")

Logistic Regression

TRAINING RESULTS
Accuracy: 0.9901278887143818
Confusion Matrix:
 [[ 567   25]
 [  19 3846]]
Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.96      0.96       592
           1       0.99      1.00      0.99      3865

    accuracy                           0.99      4457
   macro avg       0.98      0.98      0.98      4457
weighted avg       0.99      0.99      0.99      4457


TEST RESULTS
Accuracy: 0.9757847533632287
Confusion Matrix:
 [[139  16]
 [ 11 949]]
Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.90      0.91       155
           1       0.98      0.99      0.99       960

    accuracy                           0.98      1115
   macro avg       0.96      0.94      0.95      1115
weighted avg       0.98      0.98      0.98      1115




SVM

TRAINING RESULTS
Accuracy: 0.9991025353376711
Confusion Matrix:
 [[ 591    1]
 [   3 3862]]


In [239]:
with open("vectorizer.pkl", "wb") as file:
    pk.dump(vectorizer, file)

In [238]:
with open("logistic_regression_model.pkl", "wb") as file:
    pk.dump(models["Logistic Regression"], file)

In [ ]:
with open("svm_model.pkl", "wb") as file:
    pk.dump(models["SVM"], file)